In [1]:
import pandas as pd

Header modification

In [3]:
header_extract = r"C:\Users\kulareddy\Downloads\New freq load\SubscriptionHeader_04012026.xlsx"
header_df = pd.read_excel(header_extract)

In [4]:
SUB_COL = "SubscriptionNumber"
LEG_COL = "LegacyId_c"
SUFFIX = "_check"

OUT_PATH = "modified_header_df_check.xlsx"

def add_suffix_keep_na(series: pd.Series, suffix: str) -> pd.Series:
    s = series.astype("string")
    mask = s.notna() & (s != "")
    # avoid double-suffixing
    already = s.str.endswith(suffix, na=False)
    out = s.copy()
    out.loc[mask & ~already] = s.loc[mask & ~already] + suffix
    return out

# Create new columns; do NOT change existing values
header_df["New SubscriptionNumber"] = add_suffix_keep_na(header_df[SUB_COL], SUFFIX)
header_df["New LegacyId_c"] = add_suffix_keep_na(header_df[LEG_COL], SUFFIX)

# Insert new columns right after the originals
cols = header_df.columns.tolist()

cols.remove("New SubscriptionNumber")
cols.insert(cols.index(SUB_COL) + 1, "New SubscriptionNumber")

cols.remove("New LegacyId_c")
cols.insert(cols.index(LEG_COL) + 1, "New LegacyId_c")

header_df = header_df[cols]

# Export to Excel
header_df.to_excel(OUT_PATH, index=False)
print(f"Wrote: {OUT_PATH}")


Wrote: modified_header_df_check.xlsx


Line Modification

In [7]:
line_extract = r"C:\Users\kulareddy\Downloads\New freq load\SubscriptionLine_04012026.xlsx"
line_df = pd.read_excel(line_extract)

In [8]:
SUBPROD_COL = "SubscriptionProductPuid"   # e.g., U01201FCN1681006377-PRDT-1
SUB_COL     = "SubscriptionNumber"
LEG_COL     = "LegacyId_c"

SUFFIX = "_check"
OUT_PATH = "modified_line_df_check.xlsx"

def add_suffix_end(series: pd.Series, suffix: str) -> pd.Series:
    """Add suffix to end of non-empty strings; avoid double-suffixing."""
    s = series.astype("string").str.strip()
    mask = s.notna() & (s != "")
    already = s.str.endswith(suffix, na=False)
    out = s.copy()
    out.loc[mask & ~already] = s.loc[mask & ~already] + suffix
    return out

def add_suffix_before_prdt(series: pd.Series, suffix: str) -> pd.Series:
    """
    Insert suffix immediately before '-PRDT-' (first occurrence):
      U...-PRDT-1  ->  U..._NST-PRDT-1
    If '-PRDT-' not present, fall back to appending suffix at end.
    Avoid double insertion if already contains '_NST-PRDT-'.
    """
    s = series.astype("string").str.strip()
    mask = s.notna() & (s != "")

    marker = "-PRDT-"
    target = f"{suffix}{marker}"          # "_NST-PRDT-"

    has_marker = s.str.contains(marker, na=False)
    already = s.str.contains(target, na=False)

    out = s.copy()
    out.loc[mask & has_marker & ~already] = s.loc[mask & has_marker & ~already].str.replace(
        marker, target, n=1, regex=False
    )
    out.loc[mask & ~has_marker] = add_suffix_end(s.loc[mask & ~has_marker], suffix)
    return out

# Create NEW columns (original values remain unchanged)
line_df["New SubscriptionProductPuid"] = add_suffix_before_prdt(line_df[SUBPROD_COL], SUFFIX)
line_df["New SubscriptionNumber"]      = add_suffix_end(line_df[SUB_COL], SUFFIX)
line_df["New LegacyId_c"]              = add_suffix_end(line_df[LEG_COL], SUFFIX)

# Insert new columns right after originals
cols = line_df.columns.tolist()

cols.remove("New SubscriptionProductPuid")
cols.insert(cols.index(SUBPROD_COL) + 1, "New SubscriptionProductPuid")

cols.remove("New SubscriptionNumber")
cols.insert(cols.index(SUB_COL) + 1, "New SubscriptionNumber")

cols.remove("New LegacyId_c")
cols.insert(cols.index(LEG_COL) + 1, "New LegacyId_c")

line_df = line_df[cols]

# Export to Excel
line_df.to_excel(OUT_PATH, index=False)
print(f"Wrote: {OUT_PATH}")


Wrote: modified_line_df_check.xlsx


Asset Modification

In [9]:
asset_extract = r"C:\Users\kulareddy\Downloads\New freq load\SubscriptionCoveredLevel_04012026.csv"
asset_df = pd.read_csv(asset_extract)

In [10]:
SUBPROD_COL = "SubscriptionProductPuid"  # e.g., U01201FCN1681006377-PRDT-1
COV_COL     = "CoveredLevelPuid"         # assumed same pattern: <base>-PRDT-<n> (or similar)
SUFFIX = "_NST3"

OUT_PATH = "modified_asset_df_NST3.xlsx"

def add_suffix_end(series: pd.Series, suffix: str) -> pd.Series:
    s = series.astype("string").str.strip()
    mask = s.notna() & (s != "")
    already = s.str.endswith(suffix, na=False)
    out = s.copy()
    out.loc[mask & ~already] = s.loc[mask & ~already] + suffix
    return out

def add_suffix_before_prdt(series: pd.Series, suffix: str) -> pd.Series:
    marker = "-PRDT-"
    target = f"{suffix}{marker}"  # "_NST-PRDT-"

    s = series.astype("string").str.strip()
    mask = s.notna() & (s != "")

    has_marker = s.str.contains(marker, na=False)
    already = s.str.contains(target, na=False)

    out = s.copy()
    out.loc[mask & has_marker & ~already] = s.loc[mask & has_marker & ~already].str.replace(
        marker, target, n=1, regex=False
    )
    out.loc[mask & ~has_marker] = add_suffix_end(s.loc[mask & ~has_marker], suffix)
    return out

# Create NEW columns (original values remain unchanged)
asset_df["New SubscriptionProductPuid"] = add_suffix_before_prdt(asset_df[SUBPROD_COL], SUFFIX)
asset_df["New CoveredLevelPuid"]        = add_suffix_before_prdt(asset_df[COV_COL], SUFFIX)

# Insert new columns right after originals
cols = asset_df.columns.tolist()

cols.remove("New SubscriptionProductPuid")
cols.insert(cols.index(SUBPROD_COL) + 1, "New SubscriptionProductPuid")

cols.remove("New CoveredLevelPuid")
cols.insert(cols.index(COV_COL) + 1, "New CoveredLevelPuid")

asset_df = asset_df[cols]

# Export to Excel
asset_df.to_excel(OUT_PATH, index=False)
print(f"Wrote: {OUT_PATH}")


Wrote: modified_asset_df_NST3.xlsx


Bill Line Modificaion

In [ ]:
billline_extract = r"C:\Users\kulareddy\Downloads\OT n SA load\Remaining ones\BIll Line NST2.csv"
billline_df = pd.read_csv(billline_extract)

In [ ]:
COV_COL  = "CoveredLevelPuid"
BILL_COL = "BillLinePuid"

SUFFIX = "_NST2"
OUT_PATH = "modified_billline_df_NST2.xlsx"

def add_suffix_end(series: pd.Series, suffix: str) -> pd.Series:
    s = series.astype("string").str.strip()
    mask = s.notna() & (s != "")
    already = s.str.endswith(suffix, na=False)
    out = s.copy()
    out.loc[mask & ~already] = s.loc[mask & ~already] + suffix
    return out

def add_suffix_before_prdt(series: pd.Series, suffix: str) -> pd.Series:
    marker = "-PRDT-"
    target = f"{suffix}{marker}"  # "_NST-PRDT-"

    s = series.astype("string").str.strip()
    mask = s.notna() & (s != "")

    has_marker = s.str.contains(marker, na=False)
    already = s.str.contains(target, na=False)

    out = s.copy()
    out.loc[mask & has_marker & ~already] = s.loc[mask & has_marker & ~already].str.replace(
        marker, target, n=1, regex=False
    )
    out.loc[mask & ~has_marker] = add_suffix_end(s.loc[mask & ~has_marker], suffix)
    return out

# New columns (keep originals)
billline_df["New CoveredLevelPuid"] = add_suffix_before_prdt(billline_df[COV_COL], SUFFIX)
billline_df["New BillLinePuid"]     = add_suffix_before_prdt(billline_df[BILL_COL], SUFFIX)

# Place new cols right after originals
cols = billline_df.columns.tolist()

cols.remove("New CoveredLevelPuid")
cols.insert(cols.index(COV_COL) + 1, "New CoveredLevelPuid")

cols.remove("New BillLinePuid")
cols.insert(cols.index(BILL_COL) + 1, "New BillLinePuid")

billline_df = billline_df[cols]

billline_df.to_excel(OUT_PATH, index=False)
print(f"Wrote: {OUT_PATH}")
